# 課題と解答例：22_snake_pattern_features

元Notebook: [../22_snake_pattern_features.ipynb](../22_snake_pattern_features.ipynb)

## 6. 課題（実装）

1. `make_equal_area_patterns()` を作り，面積比がほぼ同じで斑点数が異なる2画像を生成する．それぞれの面積比と連結成分数を表にする．
2. `make_diagonal_touch_example()` を作り，`connectivity=4` と `connectivity=8` で連結成分数が変わる画像を生成する．
3. `noise_sensitivity(image, noise_levels, seed)` を作り，ノイズ強度ごとの面積比，連結成分数，特徴波長をDataFrameにまとめる．
4. 上の3つの関数を使い，どの特徴量が閾値やノイズに敏感かを表から説明する．

## 解答例

1. 同じ面積で個数が異なる画像は，半径の異なる円を使って作れる．

   ```python
   def make_equal_area_patterns(size=128):
       one = np.zeros((size, size), float)
       many = np.zeros((size, size), float)
       yy, xx = np.mgrid[:size, :size]
       one[(xx-size//2)**2 + (yy-size//2)**2 <= 20**2] = 1.0
       for cy in [36, 64, 92]:
           for cx in [36, 64, 92]:
               many[(xx-cx)**2 + (yy-cy)**2 <= 7**2] = 1.0
       return one, many
   ```

2. 斜め接触の例である．

   ```python
   def make_diagonal_touch_example():
       img = np.zeros((5, 5), float)
       img[1, 1] = 1
       img[2, 2] = 1
       return img
   ```

   4近傍では2成分，8近傍では1成分になる．

3. ノイズ感度は表で確認する．

   ```python
   def noise_sensitivity(image, noise_levels, seed=0):
       rng = np.random.default_rng(seed)
       rows = []
       for level in noise_levels:
           noisy = np.clip(image + level * rng.normal(size=image.shape), 0, 1)
           rows.append({
               "noise": level,
               "area": area_fraction(noisy),
               "components": component_count(noisy),
               "wavelength": dominant_wavelength(noisy),
           })
       return pd.DataFrame(rows)
   ```

4. 一般に，連結成分数は小さな孤立ノイズに敏感である．面積比は閾値近くの画素が多いと不安定になる．特徴波長は大域的な模様には比較的強いが，強いノイズではピークが崩れる．